# HeartTwin: cross-repository KIT / EDGAR diagnosis

Pinned implementation: `2b5228d9fa9a3edb195b642c6c9b4d33a6486971`. Original failed benchmark: `0b26c52a23a6b13dd41b90caacd912dfc7ab8db6`. Run in the user's Edge Colab, on CPU. This notebook creates a separate pinned environment and preserves the baseline evidence. Do not interpret a software test pass as clinical validation.

The model ablation chooses using only the original calibration samples. Archived outer scores are development reassessment, never fresh independent validation. Geometry candidates stay in Colab, retain originals, and require review. KIT source CC BY-NC 4.0; EDGAR redistribution rights and coordinate units remain unconfirmed. No raw coordinate candidates are included in the result export.

All six component repositories are pinned in the setup cell. Optional test dependencies are installed explicitly; the final environment is recorded. The process status and test logs retain failures and skips.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
BASE='0b26c52a23a6b13dd41b90caacd912dfc7ab8db6'
ROOT=Path('/content/HeartTwin_Diagnosis_20261020')
ROOT.mkdir(exist_ok=True)
OUT=ROOT/'results'; OUT.mkdir(exist_ok=True)
ENV=dict(os.environ,MPLBACKEND='Agg',PYTHONUNBUFFERED='1')
def run(command,name,timeout=1200):
    print('Starting',name,flush=True)
    with (OUT/(name+'.log')).open('w') as log:
        p=subprocess.run(list(map(str,command)),stdout=log,stderr=subprocess.STDOUT,env=ENV,timeout=timeout)
    if p.returncode:
        print((OUT/(name+'.log')).read_text()[-10000:])
        raise RuntimeError(name+' failed '+str(p.returncode))
PINS={'Virelion-HeartTwin':BASE,'Virelion-CardiMech':'5dc14d886ccbda5a1398b863f9ae7821bbffa18b','Virelion-CardiFlow':'670d370635ba09b2908f7e6736299719e86fc36b','Virelion-CardiInfer':'b4f0c3c30da5986c6a98dba6ceccf1e1de95c881','Virelion-CardiEP':'f9bfcb78cdb857e5995ce244d8645f7370d6b3bb','Virelion-CardiAnatomy':'e3cc3450c49b84334dd2da696f33a75f4f61557f','Virelion-ElectroTrace':'51adddec49efa182f9707524cb59a16b4be554d7'}
for name,pin in PINS.items():
    path=ROOT/name
    if not path.exists():
        run(['git','init',path],'init-'+name)
        run(['git','-C',path,'fetch','--depth','1','https://github.com/Virelion-Biotech/'+name+'.git',pin],'fetch-'+name)
        run(['git','-C',path,'checkout','--detach','FETCH_HEAD'],'checkout-'+name)
    assert subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True).strip()==pin
    assert not subprocess.check_output(['git','-C',str(path),'status','--porcelain','--untracked-files=no'],text=True).strip()
venv=ROOT/'venv'
if not venv.exists():run([sys.executable,'-m','venv','--without-pip',venv],'venv')
python=venv/'bin/python'
ENV['PATH']=str(python.parent)+os.pathsep+ENV['PATH']
run([sys.executable,'-m','pip','--python',python,'install','pip==25.1.1','numpy==2.2.6','scipy==1.16.3','pydantic==2.12.3','PyYAML==6.0.3','pytest==8.4.2','ruff==0.14.1','meshio==5.3.5','vtk==9.5.2','nibabel==5.3.2','pydicom==3.0.1','jsonschema==4.25.1'],'install-dependencies')
for name in PINS:run([python,'-m','pip','install','-e',ROOT/name],'install-'+name)
run([python,'-m','pip','check'],'pip-check')
(OUT/'pins.json').write_text(json.dumps(PINS,indent=2))
run([python,'-m','pip','freeze'],'environment')
repo=ROOT/'Virelion-HeartTwin'
print('DIAGNOSIS_ENVIRONMENT_READY')



In [ ]:
import json, subprocess
UNIT_PROBE = r"""
import json, numpy as np
from hearttwin.patient_closed_loop import hemodynamic_observables
from hearttwin.patient_observations import sample_prediction
import hearttwin.patient_closed_loop as module
p=hemodynamic_observables([0.,1.],np.ones((2,8)),np.ones((2,8)),np.ones((2,8)))
report={'module':module.__file__}
try:
    sample_prediction(p,{'model_output':'pressure_mmHg.LV','model_time':'time_s','time_s':[.5],'unit':'mL','prediction_unit':'mL'})
    report['wrong_unit_accepted']=True
except ValueError as error:
    report.update(wrong_unit_accepted=False,error=str(error))
print(json.dumps(report))
"""
before=subprocess.check_output([str(python),'-c',UNIT_PROBE],env=ENV,text=True)
(OUT/'prepatch_unit_reproduction.json').write_text(before)
print('PREPATCH',before)
assert json.loads(before)['wrong_unit_accepted'] is True
FIX='2b5228d9fa9a3edb195b642c6c9b4d33a6486971'
fixed_repo=ROOT/'HeartTwin-fixed'
if not fixed_repo.exists():
    run(['git','init',fixed_repo],'fixed-init')
    run(['git','-C',fixed_repo,'fetch','--depth','1','https://github.com/Virelion-Biotech/Virelion-HeartTwin.git',FIX],'fixed-fetch')
    run(['git','-C',fixed_repo,'checkout','--detach','FETCH_HEAD'],'fixed-checkout')
assert subprocess.check_output(['git','-C',str(fixed_repo),'rev-parse','HEAD'],text=True).strip()==FIX
run([python,'-m','pip','install','-e',fixed_repo],'install-fixed')
after=subprocess.check_output([str(python),'-c',UNIT_PROBE],env=ENV,text=True)
(OUT/'postpatch_unit_reproduction.json').write_text(after)
print('POSTPATCH',after)
assert json.loads(after)['wrong_unit_accepted'] is False
run([python,'-m','pytest','-q',fixed_repo/'tests/test_patient_native_output_units.py',fixed_repo/'tests/test_patient_geometry_audit.py',fixed_repo/'tests/test_patient_native_calibration.py',fixed_repo/'tests/test_patient_coupled.py','--junitxml='+str(OUT/'focused-tests.xml')],'focused-tests',timeout=600)
print((OUT/'focused-tests.log').read_text()[-2000:])
print('UNIT_FIX_AND_GEOMETRY_REGRESSIONS_COMPLETE')



In [ ]:
import json, subprocess
PATCH='2b5228d9fa9a3edb195b642c6c9b4d33a6486971'
run(['git','-C',fixed_repo,'fetch','--depth','1','https://github.com/Virelion-Biotech/Virelion-HeartTwin.git',PATCH],'patch-fetch')
run(['git','-C',fixed_repo,'checkout','--detach','FETCH_HEAD'],'patch-checkout')
assert subprocess.check_output(['git','-C',str(fixed_repo),'rev-parse','HEAD'],text=True).strip()==PATCH
baseline=repo/'validation/kit_edgar_2026_10_20'
diagnostics={}
for name,command in {
    'circulation':[python,fixed_repo/'scripts/diagnose_kit_circulation.py','--baseline-dir',baseline,'--output',OUT/'circulation'],
    'geometry':[python,fixed_repo/'scripts/diagnose_patient_geometry.py','--baseline-repo',repo,'--data-dir',ROOT/'source_data','--output',OUT/'geometry'],
    'patient-regressions':[python,fixed_repo/'scripts/validate_patient_lifecycle.py','--output',OUT/'patient-regressions','--components'],
}.items():
    log=(OUT/(name+'.log')).open('w')
    process=subprocess.Popen(list(map(str,command)),cwd=fixed_repo,env=ENV,stdout=log,stderr=subprocess.STDOUT)
    diagnostics[name]=(process,log)
    print('STARTED',name,'PID',process.pid)
(OUT/'patch_pin.json').write_text(json.dumps({'patch_commit':PATCH,'baseline':BASE,'components':PINS,'scope':'Unit regression, independent native/reference and transport checks, calibration-only model diagnosis, geometry candidates'},indent=2))
print('Diagnostics running; original benchmark evidence is unchanged.')



In [ ]:
import subprocess, json
component_runs={}
for name in ['Virelion-CardiMech','Virelion-CardiFlow','Virelion-CardiInfer','Virelion-CardiEP']:
 log=(OUT/(name+'-suite.log')).open('w')
 p=subprocess.Popen(list(map(str,[python,'-m','pytest','-q',ROOT/name/'tests','--junitxml='+str(OUT/(name+'-suite.xml'))])),cwd=ROOT/name,env=ENV,stdout=log,stderr=subprocess.STDOUT)
 component_runs[name]=(p,log)
 print('STARTED',name,p.pid)
lint_files=[fixed_repo/'hearttwin/patient_geometry_audit.py',fixed_repo/'tests/test_patient_native_output_units.py',fixed_repo/'tests/test_patient_geometry_audit.py',fixed_repo/'scripts/diagnose_kit_circulation.py',fixed_repo/'scripts/diagnose_patient_geometry.py']
lint=subprocess.run([str(python),'-m','ruff','check',*map(str,lint_files)],capture_output=True,text=True,env=ENV)
(OUT/'diagnosis-lint.log').write_text(lint.stdout+lint.stderr)
print('LINT',lint.returncode,lint.stdout,lint.stderr)
for name,(p,log) in diagnostics.items():
 print(name,'EXIT',p.poll())
 print((OUT/(name+'.log')).read_text()[-1500:])



In [ ]:
run([python,'-m','pip','install','hypothesis>=6.100','wfdb>=4.1','pyedflib>=0.1.38','skops>=0.14','flask>=3.0'],'component-test-extras')
run([python,'-m','pip','check'],'pip-check-with-extras')
run([python,'-m','pip','freeze'],'environment-with-extras')
for name in ['Virelion-CardiAnatomy','Virelion-ElectroTrace']:
 log=(OUT/(name+'-suite.log')).open('w')
 p=subprocess.Popen(list(map(str,[python,'-m','pytest','-q',ROOT/name/'tests','--junitxml='+str(OUT/(name+'-suite.xml'))])),cwd=ROOT/name,env=ENV,stdout=log,stderr=subprocess.STDOUT)
 component_runs[name]=(p,log)
 print('STARTED',name,p.pid)
for name in ['KIT_vtu_reader_diagnosis.json','KIT_frame_diagnosis.json','EDGAR_repair_diagnosis.json']:
 path=OUT/'geometry'/name
 if path.exists():
  x=json.loads(path.read_text())
  if 'surfaces' in x:
   x={k:{'before':v['before'],'after':v['after'],'removed':v['removed_zero_area_face_ids']} for k,v in x['surfaces'].items()}
  print(name,json.dumps(x))
for name,(p,log) in {**diagnostics,**component_runs}.items():
 print('STATUS',name,p.poll())
 print((OUT/(name+'.log' if name in diagnostics else name+'-suite.log')).read_text()[-2500:])



In [ ]:
for name,(process,log) in {**diagnostics,**component_runs}.items():
 print(name,'EXIT',process.poll())
 print((OUT/(name+'.log' if name in diagnostics else name+'-suite.log')).read_text()[-8000:])
for name in ['solver_transport_diagnosis.json','inner_model_selection.json','model_assumption_diagnosis.json']:
 path=OUT/'circulation'/name
 if path.exists(): print(name,path.read_text())
for name in ['KIT_vtu_reader_diagnosis.json','KIT_frame_diagnosis.json','EDGAR_repair_diagnosis.json']:
 path=OUT/'geometry'/name
 if path.exists():
  data=json.loads(path.read_text())
  if 'surfaces' in data: data={k:{'before':v['before'],'after':v['after']} for k,v in data['surfaces'].items()}
  print(name,json.dumps(data,indent=2))



In [ ]:
import base64, hashlib, zipfile
process_status={name:p.poll() for name,(p,log) in {**diagnostics,**component_runs}.items()}
(OUT/'PROCESS_STATUS.json').write_text(json.dumps(process_status,indent=2))
if any(code is None for code in process_status.values()):
 raise RuntimeError('Processes are still running; retain partial logs and rerun this cell after completion')
# Include audit statistics, never raw or derived patient coordinate arrays.
selected=[p for p in OUT.rglob('*') if p.is_file()
 and 'candidates' not in p.relative_to(OUT).parts
 and p.suffix.lower() in {'.json','.log','.xml','.md','.txt'}
 and p.name not in {'export_bundle.json','ARTIFACT_HASHES.json'}]
hashes={str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in selected}
manifest=OUT/'ARTIFACT_HASHES.json'
manifest.write_text(json.dumps(hashes,indent=2))
selected.append(manifest)
archive=ROOT/'HeartTwin_Cross_Repo_Diagnosis.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
 for p in selected:z.write(p,str(p.relative_to(OUT)))
raw=archive.read_bytes()
bundle={'archive_sha256':hashlib.sha256(raw).hexdigest(),
 'archive_git_blob_sha1':hashlib.sha1(b'blob '+str(len(raw)).encode()+b'\0'+raw).hexdigest(),
 'archive_bytes':len(raw),'archive_base64':base64.b64encode(raw).decode(),
 'text_files':{str(p.relative_to(OUT)):p.read_text() for p in selected}}
(OUT/'export_bundle.json').write_text(json.dumps(bundle))
print('PROCESS_STATUS',json.dumps(process_status))
print('ARCHIVE_SHA256',bundle['archive_sha256'])
print('ARCHIVE_BYTES',bundle['archive_bytes'])
print('Any failed process remains a failure; these are software/model diagnostics, not clinical validation.')



In [ ]:
print((OUT/'export_bundle.json').read_text())
